In [1]:
%load_ext autoreload
%autoreload 2
#%env JAX_PLATFORM_NAME=cpu
#%env JAX_ENABLE_X64=True
%matplotlib inline
#%matplotlib tk
import numpy as np
import numpy.linalg as lg
#from sklearn.model_selection import train_test_split
import jax
import jax.numpy as jnp
from util_pred import cov_mat, fit_poly_dc, N_SUBJ
from util import visSphere, bez_sph, coord_2D3D
from morphomatics.manifold import Sphere, PowerManifold
from morphomatics.stats import ExponentialBarycenter
from TimeSeries.verification_metrics import errfun
#from TimeSeries.opt import *
from TimeSeries.stats import sph_correlated_trjs, sph_rand_trjs, save_sph, load_sph
from TimeSeries.main import pred, pred_NHC, apply_nhc_filter
from TimeSeries.model import (
    VelocityEnsemble, AdaptiveVelocityEnsemble, MAEnsemble,
    Reg, RidgeReg, AR, VWA, ARMA, WeightedAverage
)
import matplotlib.pyplot as plt
from random import random
# Set random seed for reproducibility (optional)
#np.random.seed(42)

# Global constants and parameters
M = Sphere()
dist = jax.jit(M.metric.dist)
err = errfun(M.metric.dist)

def read_data(random_data=True):
    if random_data:
        # Reda Data: Random trajectories
        noise_std, mean_curve = 0.03, 'Else'  # 'Geo', 'Poly', 'Else'
        n_subj, n_points = 30, 45
        lon_max, lat_max = .75*np.pi, np.pi/20  # 4.5
        #Y, template = sph_correlated_trjs(lon_max, lat_max, n_trj=n_subj, n_points=n_points, noise_std=noise_std,mean_curve=mean_curve)  # Gauss noise
        #Y = rand_trjs(n_mat=n_subj, n_points=n_points)
        ##mean_len = int(np.mean([len(Y[i]) for i in range(len(Y))]))
        B, Y = load_sph(mean_curve)
        #B, Y_fit = fit_poly_dc(M, Y, deg=deg)
        #save_sph(B, Y, mean_curve)
        #Y = Y_fit
        n_start, n_train = 0, 20
        Y_train, B_train, Y_test = Y[n_start:n_train], B[n_start:n_train], Y[n_train:]
        W_test = []
    else:
        # Reda Data: Hur tracks
        from util_pred import load_hur
        Ex = ExponentialBarycenter()
        H = load_hur(path='../datasets/hur.npz', sph=True, wind=True)
        Y, W = [h[:, :3] for h in H], [h[:, 3] for h in H]
        #B = np.load('../datasets/deg5_coeff.npz', allow_pickle=True)['arr_0']
        B = np.load('../datasets/deg3_coeff.npz', allow_pickle=True)['arr_0']
        n_start, n_train = 197, 213
        #B, Y_fit = fit_poly_dc(M, Y, deg=deg)
        #MAE = [err(Y_fit[k], Y[k]).mae() for k in range(len(Y))]  # np.mean(MAE) =0.011
        #R2 = [err(Y_fit[k], Y[k]).r2(Ex.compute(M, Y[k], max_iter=30)) for k in range(len(Y))]  # mean: .98%
        Y_train, B_train, Y_test = Y[n_start:n_train], B[n_start:n_train], Y[n_train:]
        W_test = W[n_train:]
    return Y_train, B_train, Y_test, W_test

In [2]:
# Test setting
Y_train, B_train, Y_test, W_test = read_data(random_data=False)
n_cp = np.shape(B_train[0])[0]
deg = n_cp - 1
P = PowerManifold(M, n_cp)
n_test = len(Y_test)
Ytest = Y_test # Y_test[0:2]
AV_ensemble = VelocityEnsemble(alpha=0.5)
pred_args = {
    'n_learn': n_cp,
    'n_pred': 1,
    'iterative': True
}

In [4]:
# Regression
model = Reg(M, lag=True, degree=deg)
#Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= None)
#Y_pred, MAE_reg, metrics = predx(Ytest, model, n_learn=n_learn, n_pred=1)
#print('MAE: {:.4f}'.format(np.mean(np.concatenate(MAE_reg))))
#mae = [err(Y_pred[k], Ytest[k][n_learn:]).mae() for k in range(len(Ytest))]
#r2_val, acc, mae = validate_pred(Y_test, Y_pred, n_learn=n_learn, n_pred=n_pred)
#model.set_ensemble_strategy(ensemble)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= None)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= AV_ensemble)

MAE:              0.0165
MASE (per-track): 0.6587 ± 0.1715
Skill:            34.1%
MAE:              0.0110
MASE (per-track): 0.4761 ± 0.1246
Skill:            52.4%


In [ ]:
NHC_MAE, n_nhc_selected = apply_nhc_filter(
    metrics['MAE'], W_test, n_learn=1, n_pred=1, TC_THRESHOLD=34.0
)

all_nhc_mae = np.concatenate([nhc for nhc in NHC_MAE if len(nhc) > 0])
if len(all_nhc_mae) > 0:
    nhc_mae_mean = np.mean(all_nhc_mae)
    print(f"NHC MAE: {nhc_mae_mean:.4f}")
    print(f"Selected: {n_nhc_selected}/{metrics['n_forecasts']}")

In [ ]:
print(dist(np.array([0, 0, 1]), np.array([1, 0, 0])))

In [ ]:
# Ridge regression
# Covariance matrix and mean
Ex = ExponentialBarycenter()
mean_b_train = Ex.compute(P, B_train, max_iter=30)
cov_b_train = cov_mat(P.metric.log, B_train, mean_b_train) #+ 1e-6*np.eye(n_cp*dim)
#w = jax.vmap(jax.jit(P.metric.log), (None, 0))(mean_b_train, B_train)
#w_vec = w.reshape(n_train, -1)
#cov_b_train, shrinkage = ledoit_wolf(w_vec)
# Analyze correlation
dists_to_mean = [P.metric.dist(mean_b_train, b) for b in B_train]
eigenvalues = np.sort(lg.eigvalsh(cov_b_train))[::-1]
print(f"\nDistance to mean: {np.mean(dists_to_mean):.4f} ± {np.std(dists_to_mean):.4f}")
print(f"Eigenvalues: {eigenvalues}")
print(f"Variance explained by PC2: {eigenvalues[:2].sum()/eigenvalues.sum():.1%}")
#cov_b_train = 1e+6*cov_b_train

# mu=1e-2[1e-4, 1e-2, 1, 5, 10, 20, 40] min at mu = 5*e-2 equal 0.0301
mu = .04*1e-1
#mu = 1e-6*mu
model = RidgeReg(M, mean_b_train, cov_b_train, mu, lag=True, degree=deg)
#Y_pred_new, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= None)
Y_pred_new, metrics = pred(Ytest, model, **pred_args, ensemble_strategy= None)

In [8]:
model = AR(M, lag=True, order=2, local=True)
#model = GeometricAR(M, lag=True, order=2, convex=False, local=True, warm_start=True)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=None)
#model.ensemble_strategy = ensemble
Y_pred_new, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=AV_ensemble)

MAE:              0.0079
MASE (per-track): 0.3954 ± 0.1663
Skill:            60.5%
MAE:              0.0074
MASE (per-track): 0.3723 ± 0.1763
Skill:            62.8%


In [9]:
# Weighted averages
model = WeightedAverage(M, lag=True, order=2)
Y_pred_new, metrics = pred(Ytest, model, **pred_args)
#model.ensemble_strategy = ensemble
#Y_pred_new, metrics = pred(Ytest, model, **pred_args)

MAE:              0.0298
MASE (per-track): 1.3795 ± 0.0514
Skill:            -38.0%


In [10]:
from TimeSeries.model import WeightedAverage
# Very fast version (1-2 minutes)
model = WeightedAverage(
    M, lag=True, order=2,
    wfm_maxiter=30,  # smaller means more aggressive
    wfm_tol=1e-4,     # Less strict
    optimize_weights=True
)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=AV_ensemble)

MAE:              0.0162
MASE (per-track): 0.7548 ± 0.0398
Skill:            24.5%


In [ ]:
from TimeSeries.model import MA
model = MA(M, lag=True, order=2, use_mean_reference=False)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=AV_ensemble)

In [ ]:
model = Reg(M, lag=True, degree=deg)
MA_ensemble = MAEnsemble(alpha=1.0, order=2)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=MA_ensemble)

In [ ]:
weights = np.ones(2) / 2
MA_ensemble = MAEnsemble(alpha=0.5, order=2, weights=weights)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=MA_ensemble)

In [3]:
from TimeSeries.model import AR as AR
model = AR(M, lag=True, order=2, local=True, warm_start=True)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=None)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=AV_ensemble)

MAE:              0.0079
MASE (per-track): 0.3954 ± 0.1663
Skill:            60.5%
MAE:              0.0074
MASE (per-track): 0.3723 ± 0.1763
Skill:            62.8%


In [3]:
MA_ensemble = MAEnsemble(alpha=0.5, order=2)
model = AR(M, lag=True, order=2, local=True, warm_start=True)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=MA_ensemble)

MAE:              0.0075
MASE (per-track): 0.3879 ± 0.1799
Skill:            61.2%


In [4]:
from TimeSeries.model import ARMA
model = ARMA(M, lag=True, order=(2, 1), local=True)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=None)

MAE:              0.0079
MASE (per-track): 0.3964 ± 0.1737
Skill:            60.4%


In [5]:
model = ARMA(M, lag=True, order=(2, 1), local=True)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=AV_ensemble)

MAE:              0.0076
MASE (per-track): 0.3826 ± 0.1757
Skill:            61.7%


In [3]:
from TimeSeries.model import VWA
model = VWA(M, lag=True, order=2, local=True)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=None)
Y_pred, metrics = pred(Ytest, model, **pred_args, ensemble_strategy=AV_ensemble)

MAE:              0.0086
MASE (per-track): 0.4343 ± 0.2063
Skill:            56.6%
MAE:              0.0077
MASE (per-track): 0.3870 ± 0.1870
Skill:            61.3%


In [ ]:
k = 0
visSphere([Y_pred[k], Ytest[k]], ['r', 'b'])